# B3 multi-seed: aggregate results

Run this **after** both `B3_Multiseed_Seed2.ipynb` and `B3_Multiseed_Seed3.ipynb` have finished. Reads the three seeds' saved JSON results (seed 1 = the original `b3_training_local_refine` run, seed 2 and 3 = the two new runs) and prints mean ± std.

In [ ]:
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

import json
import numpy as np

R = '/content/drive/MyDrive/pfem_run'
run_dirs = {
    'seed1 (original)': f'{R}/b3_training_local_refine',
    'seed2': f'{R}/b3_training_local_refine_seed2',
    'seed3': f'{R}/b3_training_local_refine_seed3',
}

stress_converged, disp_combined, energy_err, reaction_err = [], [], [], []
per_seed = {}
for name, d in run_dirs.items():
    with open(f'{d}/region_local_refine_convergence.json') as f:
        region = json.load(f)
    with open(f'{d}/global_qois_and_jacobian.json') as f:
        glob = json.load(f)
    n_sub10 = next(r for r in region['sweep'] if r['n_sub'] == 10)
    stress = n_sub10['pooled_frobenius_rel_error']
    disp = glob['displacement']['mean_rel_L2_combined']
    energy = glob['energy_pooled_rms_rel']
    reaction = glob['reaction_moment_y_pooled_rms_rel']
    per_seed[name] = {'stress': stress, 'displacement': disp, 'energy': energy, 'reaction': reaction}
    stress_converged.append(stress)
    disp_combined.append(disp)
    energy_err.append(energy)
    reaction_err.append(reaction)

print('Per-seed results:')
print(json.dumps(per_seed, indent=2))

print('\n=== MEAN +/- STD ACROSS 3 SEEDS ===')
for name, vals in [('Regional Cauchy-stress (n_sub=10)', stress_converged),
                    ('Displacement (combined)', disp_combined),
                    ('Energy (pooled RMS)', energy_err),
                    ('Reaction moment (pooled RMS)', reaction_err)]:
    arr = np.array(vals)
    print(f'{name}: {arr.mean()*100:.2f}% +/- {arr.std()*100:.2f}%  (values: {[f"{v*100:.2f}%" for v in vals]})')